In [ ]:
from transformers import AutoModelForImageClassification, AutoProcessor
import torch
import torchvision.transforms as transforms
from PIL import Image
import requests
import matplotlib.pyplot as plt
import random
import numpy as np
import sys
import math

In [ ]:
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
image = Image.open("/content/sample_data/dog.bmp")

plt.imshow(image)
plt.axis("off")
plt.show

In [ ]:
from transformers import AutoImageProcessor

model_name = "hilmansw/resnet18-catdog-classifier"

model = AutoModelForImageClassification.from_pretrained(model_name).to(device)
model.eval()

# processor = AutoProcessor.from_pretrained(model_name)
# print(processor)

In [ ]:
# Les Transforms

transformTensor = transforms.ToTensor()

transformPIL = transforms.ToPILImage()

transformIn = transforms.Compose([
    transforms.ToPILImage(),
    transforms.Resize((224, 224)),  # -> je ne suis pas sûr de moi
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalisation -> ImageNet
])

transformOut = transforms.Compose([
    transforms.Normalize(
        mean=[-0.485 / 0.229, -0.456 / 0.224, -0.406 / 0.225],
        std=[1 / 0.229, 1 / 0.224, 1 / 0.225]
    )
])

In [ ]:
import torch

def pixel_to_change_generator(image, lower_pourcentage=0.001, upper_pourcentage=0.01):
    _, height, width = image.shape
    random_pourcentage = torch.FloatTensor(1).uniform_(lower_pourcentage, upper_pourcentage).to(device)
    pixel_to_change = (random_pourcentage * (height * width)).floor().int().item()

    return pixel_to_change#, random_pourcentage.item()

In [ ]:
def multiple_copies_generator(image, number_of_copies = 40):
    return image.unsqueeze(0).expand(number_of_copies, -1, -1, -1).to(device) # Format [number_of_copies, 3, x, y]

In [ ]:
# Genere aleatoirement les valeurs a ajouter sur chaque pixel

def which_pixel_generator(multiple_copies, pixel_to_change, reach = 1, targeted = False, targeted_channel = 0): # reach = 0.1  ->  -0.1 <= x <= 0.1
                                                                                       # RGB -> 0-R; 1-G; 2-B

    batch_size, channels, height, width = multiple_copies.shape

    # Set the location for the pixels to change and for each dimensions -> in64 for indexing
    pixel_c = torch.full((batch_size, pixel_to_change), targeted_channel, dtype=torch.int64).to(device) if targeted else torch.randint(0, channels, (batch_size, pixel_to_change), dtype=torch.int64).to(device)
    pixel_x = torch.randint(0, height, (batch_size, pixel_to_change), dtype=torch.int64).to(device)
    pixel_y = torch.randint(0, width, (batch_size, pixel_to_change), dtype=torch.int64).to(device)

    # Create a tensor with the values to add (+ and -)
    values_to_change = torch.empty(batch_size, pixel_to_change).uniform_(-reach, reach).to(device)

    multiple_copies = multiple_copies.clone().to(device)  # Remove if modifying original is okay
    multiple_copies[torch.arange(batch_size).unsqueeze(1), pixel_c, pixel_x, pixel_y] += values_to_change

    return multiple_copies.clamp_(0, 1)

In [ ]:
def parent_generator(multiple_copies):
    batch_size = multiple_copies.shape[0]
    number_of_parents = torch.randint(0, batch_size // 2 + 1, (1,)).item() * 2 # Random between 0 and 40 -> even
                                    # Start from 1 to avoid iterations without crossover

    parents_index = torch.randperm(batch_size, device=device)[:number_of_parents]  # shuffle the indexes and take number_of_parents indexes

    return parents_index

In [ ]:
def crossover_generator(multiple_copies, parents_index, pourcentage_height = 0.15):
  multiple_copies_cross = multiple_copies.clone().to(device)

  batch_size, channels, height, width = multiple_copies.shape

  for i in range(0, len(parents_index), 2):
    parent_index_1 = parents_index[i]
    parent_index_2 = parents_index[i + 1]

    max_size_square = int(multiple_copies.shape[3] * pourcentage_height)  # % hauteur de l'image -> multiple_copies[0].shape[2] prend le nombre de pixel d'une image

    # Swaping zone setting
    which_channel = torch.randint(0, channels, (1,)).to(device)
    size_height = torch.randint(0, max_size_square + 1, (1,)).to(device)
    start_height = torch.randint(0, height - size_height + 1, (1,)).to(device)
    size_width = torch.randint(0, max_size_square + 1, (1,)).to(device)
    start_width = torch.randint(0, width - size_width + 1, (1,)).to(device)

    # Swaping
    swap_zone = multiple_copies_cross[parent_index_1, which_channel, start_height:start_height + size_height, start_width:start_width + size_width,].clone()

    multiple_copies_cross[parent_index_1, which_channel, start_height:start_height + size_height, start_width:start_width + size_width] = \
        multiple_copies_cross[parent_index_2, which_channel, start_height:start_height + size_height, start_width:start_width + size_width]

    multiple_copies_cross[parent_index_2, which_channel, start_height:start_height + size_height, start_width:start_width + size_width] = swap_zone

  return multiple_copies_cross

In [ ]:
def through_model(multiple_copies):
    # multiple_copies.shape -> (40, 3, 225, 224)
    batch_size = multiple_copies.shape[0]

    # One pic after the other, transformer don't handle 4D
    input_copies = torch.stack([transformIn(multiple_copies[i]).to(device) for i in range(batch_size)])

    with torch.no_grad():  # CNN inputs
        outputs = model(input_copies)

    processed_copies = transformOut(input_copies)

    logits = outputs.logits  # logits -> [40, num_classes]
    probabilities = torch.nn.functional.softmax(logits, dim=-1)

    return probabilities


In [ ]:
def selection(multiple_copies, probabilities, elites, wanted_class, unwanted_class, wanted=True):

    batch_size, channels, height, width = multiple_copies.shape

    probabilities_class = probabilities[:, wanted_class] if wanted else probabilities[:, unwanted_class]
    probabilities_class = probabilities_class.to(device)

    sorted_probabilities, indices = torch.sort(probabilities_class, descending=wanted)
    sorted_probabilities = sorted_probabilities.to(device)
    indices = indices.to(device)

    elites_index = indices[:elites]
    middle_index = indices[elites:batch_size//2]
    low_index = indices[batch_size//2:]

    elites_proba = sorted_probabilities[:elites]
    middle_proba = sorted_probabilities[elites:batch_size//2]
    low_proba = sorted_probabilities[batch_size//2:]

    elites_proba = elites_proba.float()
    middle_proba = middle_proba.float()
    low_proba = low_proba.float()

    elites_selection = multiple_copies[elites_index].to(device)
    middle_selection = multiple_copies[middle_index].to(device)
    low_selection = multiple_copies[low_index].to(device)

    return elites_selection, middle_selection, low_selection, elites_index, elites_proba

In [ ]:
def from_rgb_to_ycbcr(tensor_image):
    transform_matrix = torch.tensor([[0.299, 0.587, 0.114],
                                     [-0.169, -0.331, 0.499],
                                     [0.499, -0.460, -0.039]], device=device)

    tensor_image_ycbcr = torch.matmul(transform_matrix, tensor_image.view(3, -1)).to(device)

    # Reshape to have the original pytorch shape for images
    tensor_image_ycbcr = tensor_image_ycbcr.view(3, tensor_image.shape[1], tensor_image.shape[2])

    return tensor_image_ycbcr

In [ ]:
def checker(elites_index, elites_proba, threshold):

    # Search uniquely in elites
    for i in range(len(elites_proba)):
      if elites_proba[i] >= threshold:
        return True, elites_index[i]
      else:
        return False, -1

In [ ]:
# ⚙️ HYPERPARAMETERS
ENUMS = 100

WANTED_CLASS = 0
UNWANTED_CLASS = 1 # not used

HEIGHT = 0.15 # significantly increse running time
REACH = 0.5 # value added to each pixel -> -REACH <= x <= REACH
MAX_POURCENTAGE = 0.2 # number of pixels changed = MAX_POURCENTAGE * total number of pixel -> 0.2 = 20% changed
ELITES = 10

THRESHOLD = 0.6 # sucess if the image is considered (THRESHOLD*10)% of the wanted class -> 0.6 = 60% of the wanted class

best_diff = sys.maxsize

# 🧑‍🎨 LOAD AND PROCESS THE BASE IMAGE
image = Image.open("/content/sample_data/dog.bmp")

tensor_image = transformTensor(image).to(device)
multiple_copies = multiple_copies_generator(tensor_image, 40)

pixel_to_change = pixel_to_change_generator(tensor_image, upper_pourcentage=MAX_POURCENTAGE)

multiple_copies = which_pixel_generator(multiple_copies, pixel_to_change, reach=REACH)

# 🤖 MODEL AND SELECTION
probabilities = through_model(multiple_copies)
elites_selection, middle_selection, low_selection, elites_index, elites_proba = selection(multiple_copies, probabilities, ELITES, WANTED_CLASS, UNWANTED_CLASS, wanted=True)

predicted_classes = torch.argmax(probabilities, dim=-1) # return the classes that the model predicted for each image
print(predicted_classes)

for i in range(ENUMS):

  # 👨‍💻 COMPUTE EACH SPLIT (20-10-10) => 40

  # 20 index
  top_selection = torch.cat((elites_selection, middle_selection))
  # 10 index
  random_low_selection = low_selection[torch.randperm(len(low_selection))[:10]] # permutation alea. des index puis on selectionne simplement
  # 10 index
  parents_index = parent_generator(top_selection)
  crossed_copies = crossover_generator(top_selection, parents_index, pourcentage_height=HEIGHT)

  # 40 index
  multiple_copies = torch.cat((top_selection, random_low_selection, crossed_copies)) # concatenate the splits together

  # ❗️Now we have the final image set for this iteration to pass through the model!

  # 🤖 MODEL AND SELECTION
  probabilities = through_model(multiple_copies)
  elites_selection, middle_selection, low_selection, elites_index, elites_proba = selection(multiple_copies, probabilities, ELITES, WANTED_CLASS, UNWANTED_CLASS, wanted=True)

  # 👍 RESULTS FOUND?/ HOW GOOD ARE THEY?
  success, success_index = checker(elites_index, elites_proba, THRESHOLD)
  if success:
    print("Adversarial image found!")
    adv_image = multiple_copies[success_index]

    # 🥸 DISPLAY BEST IMAGE
    image_restored = transformPIL(adv_image)
    plt.imshow(image_restored)
    plt.axis("off")
    plt.show

    break